# Course 01: Introduction to AI and ML on Google Cloud

**Companion notebook for**: `01-intro-ai-ml-gcp.html`

## Learning Objectives
- Set up the Google Cloud AI Platform SDK (`google-cloud-aiplatform`)
- Explore the Vertex AI Model Garden programmatically
- Call pre-trained Google Cloud ML APIs (Vision, NLP, Translation)
- Understand Vertex AI AutoML concepts via SDK code
- Interact with a Vertex AI prediction endpoint
- Call the Gemini API via `vertexai.generative_models`

## Prerequisites
- A Google Cloud project with billing enabled (for API calls)
- `GCP_PROJECT_ID` environment variable set, OR replace `your-project-id` below
- Python 3.10+

---
## Setup & Dependencies

In [2]:
%pip install -q google-cloud-aiplatform google-cloud-bigquery google-cloud-language google-cloud-vision google-cloud-speech google-cloud-videointelligence google-cloud-translate vertexai


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json

# --- Project Configuration ---
# Set your GCP project ID here or via environment variable
PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
LOCATION = "us-central1"  # Default region for Vertex AI

print(f"Project ID: {PROJECT_ID}")
print(f"Location:   {LOCATION}")

if PROJECT_ID == "your-project-id":
    print("\n⚠️  WARNING: Set GCP_PROJECT_ID or replace 'your-project-id' above.")
    print("   Some cells require a real GCP project with billing enabled.")

Project ID: your-project-id
Location:   us-central1

⚠️  WARNING: Set GCP_PROJECT_ID or replace 'your-project-id' above.
   Some cells require a real GCP project with billing enabled.


In [ ]:
# Authenticate in Colab (skip if running locally with gcloud auth)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
except ImportError:
    print("Not running in Colab — assuming local gcloud auth.")
    print("Run: gcloud auth application-default login")

---
## Section 1: Initialize Vertex AI SDK

The `google-cloud-aiplatform` SDK is the primary Python interface for Vertex AI.
You must initialize it with your project and location before using any Vertex AI services.

In [ ]:
from google.cloud import aiplatform

# Initialize the Vertex AI SDK
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
)

print(f"Vertex AI SDK initialized.")
print(f"  Project:  {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  SDK version: {aiplatform.__version__}")

---
## Section 2: Explore Model Garden

The **Model Garden** is Vertex AI's catalog of foundation models — Google's own models
(Gemini, PaLM, Imagen), open-source models (Llama, Mistral), and partner models.

We can list available publisher models programmatically using the SDK.

In [ ]:
# REQUIRES: GCP project with billing enabled
# List available foundation models in Model Garden

from google.cloud.aiplatform import models as aiplatform_models

# List publisher models (foundation models in Model Garden)
try:
    publisher_models = aiplatform.Model.list(
        filter='labels.is_publisher_model=true',
        order_by='display_name',
    )
    print(f"Found {len(publisher_models)} models in your project's Model Registry.\n")
    for model in publisher_models[:10]:
        print(f"  - {model.display_name} (resource: {model.resource_name})")
except Exception as e:
    print(f"Could not list models: {e}")
    print("\nThis is expected if no models are deployed yet.")
    print("Visit: https://console.cloud.google.com/vertex-ai/model-garden")

In [ ]:
# --- Demo: Exploring Model Garden categories (no API call needed) ---
# This is what you would find in the Model Garden UI

model_garden_categories = {
    "Google Foundation Models": [
        "gemini-1.5-pro — Multimodal (text, image, video, audio)",
        "gemini-1.5-flash — Fast multimodal, lower cost",
        "gemini-2.0-flash — Latest generation, multimodal",
        "text-embedding-005 — Text embeddings (768-dim)",
        "imagen-3.0-generate — Image generation",
        "chirp-2 — Speech-to-text",
        "codey — Code generation and completion",
    ],
    "Open-Source Models": [
        "meta/llama-3.1-405b-instruct — Meta's Llama 3.1",
        "mistralai/mistral-7b-instruct — Mistral 7B",
        "google/flan-t5-xxl — Encoder-decoder, instruction-tuned",
        "google/gemma-2-27b-it — Google's open model",
    ],
    "Partner Models": [
        "anthropic/claude-3.5-sonnet — Anthropic Claude",
        "ai21/jamba-1.5-large — AI21 Jamba",
        "cohere/command-r-plus — Cohere Command R+",
    ],
}

print("=" * 60)
print("Vertex AI Model Garden — Key Models")
print("=" * 60)

for category, models_list in model_garden_categories.items():
    print(f"\n📂 {category}")
    print("-" * 50)
    for m in models_list:
        print(f"   • {m}")

---
## Section 3: Pre-trained APIs — Cloud Vision

The **Cloud Vision API** analyzes images and returns structured annotations.
No training needed — send an image, get labels, faces, text, landmarks, etc.

This is a **Tier 1** (pre-trained API) solution — the simplest GCP AI option.

In [ ]:
# REQUIRES: GCP project with billing enabled + Vision API enabled
from google.cloud import vision

def detect_labels_from_uri(image_uri: str, max_results: int = 10) -> list:
    """
    Call Cloud Vision API to detect labels in an image.
    
    Args:
        image_uri: A publicly accessible image URL or GCS URI (gs://...)
        max_results: Maximum number of labels to return
    
    Returns:
        List of (description, score) tuples
    """
    client = vision.ImageAnnotatorClient()
    
    image = vision.Image()
    image.source.image_uri = image_uri
    
    response = client.label_detection(
        image=image,
        max_results=max_results,
    )
    
    if response.error.message:
        raise Exception(f"Vision API error: {response.error.message}")
    
    labels = [
        (label.description, round(label.score, 3))
        for label in response.label_annotations
    ]
    return labels


# --- Example: Detect labels in a sample image ---
sample_image = "https://storage.googleapis.com/cloud-samples-data/vision/label/wakeupcat.jpg"

try:
    labels = detect_labels_from_uri(sample_image)
    print(f"Labels detected in: {sample_image}\n")
    print(f"{'Label':<30} {'Confidence':>10}")
    print("-" * 42)
    for desc, score in labels:
        bar = '█' * int(score * 20)
        print(f"{desc:<30} {score:>8.1%}  {bar}")
except Exception as e:
    print(f"Vision API call failed: {e}")
    print("\nMock output (what you would see):")
    mock_labels = [
        ("Cat", 0.983), ("Felidae", 0.961), ("Small to medium-sized cats", 0.945),
        ("Whiskers", 0.932), ("Carnivore", 0.891), ("Tabby cat", 0.867),
        ("Snout", 0.843), ("Fur", 0.821), ("Domestic short-haired cat", 0.798),
    ]
    print(f"\n{'Label':<30} {'Confidence':>10}")
    print("-" * 42)
    for desc, score in mock_labels:
        bar = '█' * int(score * 20)
        print(f"{desc:<30} {score:>8.1%}  {bar}")

---
## Section 4: Pre-trained APIs — Cloud Natural Language

The **Cloud Natural Language API** analyzes text for:
- **Sentiment** — Overall feeling (positive/negative) with magnitude
- **Entities** — Named entities (people, places, organizations)
- **Syntax** — Part-of-speech tags, dependency parse trees
- **Classification** — Content categories (700+ categories)

In [ ]:
# REQUIRES: GCP project with billing enabled + NL API enabled
from google.cloud import language_v1


def analyze_sentiment(text: str) -> dict:
    """
    Analyze sentiment of text using Cloud Natural Language API.
    
    Returns:
        dict with 'score' (-1 to 1) and 'magnitude' (0 to inf)
        score: overall sentiment (negative to positive)
        magnitude: strength of emotion (can be high for mixed sentiment)
    """
    client = language_v1.LanguageServiceClient()
    
    document = language_v1.Document(
        content=text,
        type_=language_v1.Document.Type.PLAIN_TEXT,
    )
    
    response = client.analyze_sentiment(document=document)
    sentiment = response.document_sentiment
    
    return {
        "score": round(sentiment.score, 3),
        "magnitude": round(sentiment.magnitude, 3),
        "sentences": [
            {
                "text": s.text.content,
                "score": round(s.sentiment.score, 3),
            }
            for s in response.sentences
        ],
    }


# --- Example: Analyze sentiment of product reviews ---
reviews = [
    "This product is absolutely fantastic! Best purchase I've made all year.",
    "Terrible quality. Broke after two days. Complete waste of money.",
    "It's okay. Nothing special but gets the job done for the price.",
    "The AI features are incredible but the battery life is disappointing.",
]

try:
    print(f"{'Review (truncated)':<55} {'Score':>7} {'Magnitude':>10} {'Label':>10}")
    print("-" * 88)
    for review in reviews:
        result = analyze_sentiment(review)
        label = "Positive" if result['score'] > 0.2 else "Negative" if result['score'] < -0.2 else "Neutral"
        print(f"{review[:53]:<55} {result['score']:>7.3f} {result['magnitude']:>10.3f} {label:>10}")
except Exception as e:
    print(f"NL API call failed: {e}")
    print("\nMock output (what you would see):")
    mock_results = [
        ("This product is absolutely fantastic! Best purchase...",  0.900, 1.800, "Positive"),
        ("Terrible quality. Broke after two days. Complete wa...", -0.800, 1.600, "Negative"),
        ("It's okay. Nothing special but gets the job done f...",  0.100, 0.400, "Neutral"),
        ("The AI features are incredible but the battery lif...",  0.100, 1.200, "Neutral"),
    ]
    print(f"\n{'Review (truncated)':<55} {'Score':>7} {'Magnitude':>10} {'Label':>10}")
    print("-" * 88)
    for text, score, mag, label in mock_results:
        print(f"{text:<55} {score:>7.3f} {mag:>10.3f} {label:>10}")

### Understanding Sentiment Scores

| Score Range | Meaning | Magnitude Interpretation |
|---|---|---|
| **0.25 to 1.0** | Positive | Higher = more emotional content |
| **-0.25 to 0.25** | Neutral / Mixed | High magnitude + neutral score = mixed sentiment |
| **-1.0 to -0.25** | Negative | Higher = more emotional content |

**Key insight**: A score near 0 with high magnitude means **mixed** sentiment (both positive and negative), not indifference.

---
## Section 5: Vertex AI AutoML — Concepts & SDK Code

AutoML lets you train custom models by providing labeled data. Vertex AI handles:
- Architecture search (NAS)
- Feature engineering
- Hyperparameter tuning
- Model selection and ensembling

The following code shows the SDK pattern. **Actual training is commented out** because it
requires billing and takes 1-8 hours depending on dataset size.

In [ ]:
# REQUIRES: GCP project with billing enabled
# This cell demonstrates the AutoML SDK pattern — commented out to avoid costs.

from google.cloud import aiplatform

# Step 1: Create a managed dataset
# AutoML requires a Vertex AI Dataset pointing to your labeled data
"""
# --- Image Classification Example ---
dataset = aiplatform.ImageDataset.create(
    display_name="plant-disease-dataset",
    gcs_source="gs://my-bucket/plant-images/import.csv",
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.single_label_classification,
)
print(f"Dataset created: {dataset.resource_name}")

# Step 2: Train an AutoML model
job = aiplatform.AutoMLImageTrainingJob(
    display_name="plant-disease-classifier",
    prediction_type="classification",
    multi_label=False,
    model_type="CLOUD",  # CLOUD = hosted on Google, EDGE = for edge devices
)

model = job.run(
    dataset=dataset,
    model_display_name="plant-disease-v1",
    training_fraction_split=0.8,
    validation_fraction_split=0.1,
    test_fraction_split=0.1,
    budget_milli_node_hours=8000,  # Max training budget (8 node-hours)
)
print(f"Model trained: {model.resource_name}")

# Step 3: Deploy to an endpoint
endpoint = model.deploy(
    deployed_model_display_name="plant-disease-endpoint",
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=3,
)
print(f"Endpoint: {endpoint.resource_name}")
"""

print("AutoML SDK pattern (commented out to avoid costs):")
print("")
print("  1. aiplatform.ImageDataset.create(gcs_source=...)")
print("  2. aiplatform.AutoMLImageTrainingJob(...).run(dataset=...)")
print("  3. model.deploy(machine_type=..., min_replica_count=...)")
print("")
print("Supports: Image, Text, Tabular, Video classification/regression")
print("Typical training time: 1-8 hours")
print("Typical cost: $3-50 per node-hour")

In [ ]:
# --- AutoML Tabular Example (also commented out) ---
"""
# Tabular AutoML is common for structured/CSV data
dataset = aiplatform.TabularDataset.create(
    display_name="customer-churn-dataset",
    gcs_source="gs://my-bucket/churn-data/train.csv",
)

job = aiplatform.AutoMLTabularTrainingJob(
    display_name="churn-prediction",
    optimization_prediction_type="classification",
    optimization_objective="maximize-au-prc",  # Area under PR curve
)

model = job.run(
    dataset=dataset,
    target_column="churned",
    training_fraction_split=0.8,
    validation_fraction_split=0.1,
    test_fraction_split=0.1,
    budget_milli_node_hours=1000,  # 1 node-hour budget
    model_display_name="churn-model-v1",
)
"""

print("AutoML Tabular pattern:")
print("  - Optimization objectives: maximize-au-roc, maximize-au-prc,")
print("    minimize-log-loss, minimize-rmse, minimize-mae")
print("  - Supports: classification, regression, forecasting")
print("  - Automatically handles missing values, encoding, feature selection")

---
## Section 6: Vertex AI Prediction Endpoint

Once a model is deployed to a **Vertex AI Endpoint**, you can send prediction requests.
The endpoint handles autoscaling, load balancing, and versioning.

In [ ]:
# REQUIRES: A deployed model endpoint
# Replace ENDPOINT_ID with your actual endpoint ID

ENDPOINT_ID = "YOUR_ENDPOINT_ID"  # e.g., "1234567890123456789"

"""
# --- Online Prediction ---
endpoint = aiplatform.Endpoint(ENDPOINT_ID)

# For tabular models, send feature values as a list of dicts
instances = [
    {
        "age": 35,
        "monthly_charges": 75.50,
        "tenure_months": 24,
        "contract_type": "month-to-month",
        "internet_service": "fiber_optic",
    },
]

predictions = endpoint.predict(instances=instances)
print(f"Predictions: {predictions.predictions}")
print(f"Model ID:    {predictions.deployed_model_id}")
"""

# --- Demo: What prediction response looks like ---
mock_prediction = {
    "predictions": [
        {
            "classes": ["not_churned", "churned"],
            "scores": [0.73, 0.27],
        }
    ],
    "deployed_model_id": "1234567890",
    "model_version_id": "1",
}

print("Example prediction response:")
print(json.dumps(mock_prediction, indent=2))
print("\nKey fields:")
print("  - predictions[].classes: Label names")
print("  - predictions[].scores: Confidence per class (sum to 1.0)")
print("  - deployed_model_id: Which model version served this request")

In [ ]:
# --- Batch Prediction (for large-scale offline inference) ---
"""
# Batch prediction processes a GCS file of instances
batch_job = model.batch_predict(
    job_display_name="churn-batch-predict",
    gcs_source="gs://my-bucket/batch-input.jsonl",
    gcs_destination_prefix="gs://my-bucket/batch-output/",
    machine_type="n1-standard-4",
    starting_replica_count=2,
    max_replica_count=10,
)
batch_job.wait()  # Blocks until complete
print(f"Output: {batch_job.output_info}")
"""

print("Batch Prediction pattern:")
print("  - Input: JSONL file in GCS (one instance per line)")
print("  - Output: JSONL file in GCS with predictions")
print("  - Use for: Large-scale offline scoring, nightly pipelines")
print("  - Advantage: No always-on endpoint needed (cost-effective)")

---
## Section 7: Gemini API via Vertex AI

**Gemini** is Google's frontier multimodal model. Access it through the `vertexai.generative_models`
module for text generation, multimodal understanding, and structured output.

In [ ]:
# REQUIRES: GCP project with Vertex AI API enabled
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

# Initialize Vertex AI for generative models
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Load the Gemini model
model = GenerativeModel("gemini-1.5-flash")

print(f"Model loaded: gemini-1.5-flash")
print(f"Project: {PROJECT_ID}")

In [ ]:
# --- Simple text generation ---
# REQUIRES: GCP project with billing enabled

prompt = """You are a Google Cloud expert. Explain the difference between 
AutoML and custom training on Vertex AI in 3 concise bullet points. 
Format each bullet as: **Topic**: explanation."""

try:
    response = model.generate_content(
        prompt,
        generation_config=GenerationConfig(
            temperature=0.2,
            max_output_tokens=500,
            top_p=0.8,
        ),
    )
    print("Gemini Response:")
    print(response.text)
    print(f"\n--- Metadata ---")
    print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
    print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
except Exception as e:
    print(f"Gemini API call failed: {e}")
    print("\nMock response (what you would see):")
    print("")
    print("• **Control**: AutoML handles architecture search, feature engineering,")
    print("  and hyperparameter tuning automatically. Custom training gives you full")
    print("  control over model code, framework choice, and training loop.")
    print("")
    print("• **Expertise**: AutoML requires no ML expertise — just labeled data.")
    print("  Custom training requires data scientists who can write training code")
    print("  in TensorFlow, PyTorch, or scikit-learn.")
    print("")
    print("• **Use Case**: Choose AutoML when speed-to-production matters and your")
    print("  problem fits standard patterns. Choose custom training when you need")
    print("  novel architectures, custom loss functions, or multi-GPU training.")

In [ ]:
# --- Chat conversation with Gemini ---
# REQUIRES: GCP project with billing enabled

try:
    chat = model.start_chat()
    
    messages = [
        "What is Vertex AI Feature Store and why is it important?",
        "How does it prevent training-serving skew?",
        "Give me a one-line gcloud command to create a feature store.",
    ]
    
    for msg in messages:
        print(f"\n👤 User: {msg}")
        response = chat.send_message(
            msg,
            generation_config=GenerationConfig(
                temperature=0.3,
                max_output_tokens=300,
            ),
        )
        print(f"\n🤖 Gemini: {response.text}")
        print("-" * 60)

except Exception as e:
    print(f"Chat failed: {e}")
    print("\nThis requires a GCP project with billing and Vertex AI API enabled.")

---
## Section 8: GCP Service Decision Matrix

This is the key mental model for the exam: given a scenario, which service do you recommend?

In [ ]:
# Decision matrix — use this as a study reference

decision_matrix = [
    {
        "scenario": "Extract text from scanned documents",
        "answer": "Document AI / Vision API (OCR)",
        "tier": "Pre-trained API",
        "reason": "Common task, no training data needed",
    },
    {
        "scenario": "Classify customer support tickets by category",
        "answer": "AutoML Text (if custom categories) or NL API (if standard)",
        "tier": "AutoML / Pre-trained",
        "reason": "NL API has built-in classification; AutoML for custom labels",
    },
    {
        "scenario": "Predict churn from BigQuery data (SQL team)",
        "answer": "BigQuery ML",
        "tier": "BigQuery ML",
        "reason": "Data in BQ, team knows SQL, no infra setup",
    },
    {
        "scenario": "Build a custom object detector for manufacturing defects",
        "answer": "AutoML Image (object detection)",
        "tier": "AutoML",
        "reason": "Domain-specific, labeled data available, no ML expertise",
    },
    {
        "scenario": "Train a custom transformer model on proprietary data",
        "answer": "Vertex AI Custom Training with GPUs/TPUs",
        "tier": "Custom Training",
        "reason": "Custom architecture, full control needed",
    },
    {
        "scenario": "Build a chatbot grounded in company documentation",
        "answer": "Vertex AI Search / Gemini + RAG",
        "tier": "Foundation Model",
        "reason": "Grounding with retrieval, not fine-tuning",
    },
    {
        "scenario": "Translate product listings to 20 languages",
        "answer": "Cloud Translation API (Basic or Advanced)",
        "tier": "Pre-trained API",
        "reason": "Common task, 100+ languages supported",
    },
    {
        "scenario": "Real-time speech transcription for call center",
        "answer": "Cloud Speech-to-Text (streaming)",
        "tier": "Pre-trained API",
        "reason": "Streaming recognition, speaker diarization",
    },
]

print("GCP ML Service Decision Matrix")
print("=" * 90)
print(f"{'Scenario':<50} {'Answer':<30} {'Tier'}")
print("-" * 90)
for row in decision_matrix:
    print(f"{row['scenario']:<50} {row['answer']:<30} {row['tier']}")
    print(f"  └─ Why: {row['reason']}")
    print()

---
## Summary

In this notebook we covered the core concepts and SDK patterns for Course 01:

1. **Vertex AI SDK Setup** — `aiplatform.init(project=..., location=...)` initializes everything.

2. **Model Garden** — Discover foundation models (Google, open-source, partner) through the SDK or console.

3. **Pre-trained APIs** — Cloud Vision for image labels, Cloud NLP for sentiment analysis. No training needed.

4. **AutoML** — Provide labeled data, Vertex AI trains a custom model. SDK pattern: `Dataset.create()` → `TrainingJob.run()` → `model.deploy()`.

5. **Prediction Endpoints** — Online (real-time) and batch prediction via `endpoint.predict()` or `model.batch_predict()`.

6. **Gemini API** — `GenerativeModel("gemini-1.5-flash")` for text generation and chat via `vertexai.generative_models`.

**Next notebook**: Course 02 covers data preparation for ML APIs — Dataflow, Dataproc, Dataprep, and deep dives into Vision, NLP, Speech, Translation, and Video Intelligence APIs.